# 07_Requested_demo_Broken_report_value_anomaly

Detect broken Power BI reports by analyzing numerical anomalies in visual values.

**Approach:**
1. Extract DAX queries directly from report visuals (no user-supplied DAX)
2. Run each visual's DAX against the semantic model to get baseline values
3. Compare report-displayed values vs model baseline values
4. Flag anomalies: blanks, zeros, abnormally high values, or values outside tolerance

This avoids the problem of users writing incorrect DAX queries or mismatched contexts.

In [ ]:
%pip install semantic-link-labs pandas

In [ ]:
import json
import math
import re
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import sempy_labs as labs
from sempy_labs.report import get_report_json, launch_report
from IPython.display import display

# ── Parameters ────────────────────────────────────────────────────────────────
report_name = ""  # Power BI report name
report_workspace = None  # Report workspace (or None for default)

dataset_name = ""  # Semantic model to validate against
dataset_workspace = None  # Semantic model workspace (or None if same as report)

# Anomaly detection thresholds (applied globally to all visuals)
tolerance_pct = 0.05  # 5% tolerance
tolerance_abs = 100.0  # absolute minimum tolerance
high_multiplier = 1.5  # flag if report value > baseline * 1.5
allow_blank = False
allow_zero = False
# ──────────────────────────────────────────────────────────────────────────────

# Render the report inline so you can see the current visual state.
launch_report(report=report_name, workspace=report_workspace)

# Retrieve report JSON and visual definitions.
report_json = get_report_json(report=report_name, workspace=report_workspace)


def _extract_dax_from_visual(visual_obj: dict) -> Optional[str]:
    """
    Extract DAX query from a visual object.
    Looks for 'config.query' which contains the DAX/M expression.
    """
    if not isinstance(visual_obj, dict):
        return None

    config = visual_obj.get("config", {})
    if isinstance(config, str):
        try:
            config = json.loads(config)
        except (json.JSONDecodeError, ValueError):
            return None

    query = config.get("query")
    if isinstance(query, str):
        return query
    elif isinstance(query, dict):
        # Sometimes query is nested; check for 'Commands' or similar
        return json.dumps(query)

    return None


def _extract_visual_value_from_display(visual_obj: dict) -> Optional[float]:
    """
    Attempt to extract the displayed numeric value from a visual.
    This is a heuristic - looks for common value locations in visual config.
    """
    if not isinstance(visual_obj, dict):
        return None

    # Check common locations
    candidates = []
    for key in ["dataValues", "values", "data", "displayValue"]:
        val = visual_obj.get(key)
        if val is not None:
            candidates.append(val)

    # Also check config
    config = visual_obj.get("config", {})
    if isinstance(config, str):
        try:
            config = json.loads(config)
        except (json.JSONDecodeError, ValueError):
            pass
    if isinstance(config, dict):
        for key in ["dataValues", "values", "data"]:
            val = config.get(key)
            if val is not None:
                candidates.append(val)

    # Try to extract first numeric from candidates
    for cand in candidates:
        if isinstance(cand, (int, float)) and not isinstance(cand, bool):
            return float(cand)
        if isinstance(cand, list) and cand and isinstance(cand[0], (int, float)):
            return float(cand[0])

    return None


def _run_scalar_dax(
    dataset: str, dax_query: str, workspace: Optional[str]
) -> Optional[float]:
    """
    Execute a DAX query and extract the first numeric value from the result.
    """
    try:
        result = labs.evaluate_dax_impersonation(
            dataset=dataset,
            dax_query=dax_query,
            workspace=workspace,
        )
        if isinstance(result, pd.DataFrame):
            if result.empty:
                return None
            row = result.iloc[0]
            for val in row.tolist():
                if pd.isna(val):
                    continue
                if isinstance(val, (int, float)) and not isinstance(val, bool):
                    return float(val)
                try:
                    return float(val)
                except (TypeError, ValueError):
                    continue
        return None
    except Exception as e:
        print(f"Error executing DAX: {e}")
        return None


def _is_blank(value: Optional[float]) -> bool:
    return value is None or (isinstance(value, float) and math.isnan(value))


def _is_zero(value: Optional[float], zero_epsilon: float = 1e-12) -> bool:
    if _is_blank(value):
        return False
    return abs(value) <= zero_epsilon


# Extract all visuals and their DAX queries from the report.
visuals_data = []

# Walk report JSON to find visual definitions
def _extract_visuals(obj, visuals_list=None):
    if visuals_list is None:
        visuals_list = []

    if isinstance(obj, dict):
        # Check if this is a visual object (has common visual properties)
        if "config" in obj or "query" in obj:
            visuals_list.append(obj)
        for v in obj.values():
            _extract_visuals(v, visuals_list)

    elif isinstance(obj, list):
        for item in obj:
            _extract_visuals(item, visuals_list)

    elif isinstance(obj, str):
        # Handle double-serialized JSON
        if len(obj) > 2 and obj[0] in ("{", "["):
            try:
                _extract_visuals(json.loads(obj), visuals_list)
            except (json.JSONDecodeError, ValueError):
                pass

    return visuals_list


visuals = _extract_visuals(report_json)
print(f"Extracted {len(visuals)} visual(s) from report.\n")

rows = []
for idx, visual in enumerate(visuals, start=1):
    dax_query = _extract_dax_from_visual(visual)
    if not dax_query:
        print(f"Visual {idx}: No DAX query found, skipping.")
        continue

    # Run DAX on model to get baseline
    baseline_value = _run_scalar_dax(dataset_name, dax_query, dataset_workspace)

    # Try to extract displayed value from visual
    report_value = _extract_visual_value_from_display(visual)

    # Evaluate anomalies
    blank_flag = _is_blank(report_value) and not allow_blank
    zero_flag = _is_zero(report_value) and not allow_zero

    high_flag = False
    delta_flag = False
    tolerance_used = None
    delta_value = None

    if not _is_blank(report_value) and not _is_blank(baseline_value):
        if baseline_value != 0:
            high_flag = report_value > (baseline_value * high_multiplier)
        delta_value = abs(report_value - baseline_value)
        tolerance_used = max(tolerance_abs, abs(baseline_value) * tolerance_pct)
        delta_flag = delta_value > tolerance_used

    is_broken = blank_flag or zero_flag or high_flag or delta_flag

    rows.append(
        {
            "visual_idx": idx,
            "report_value": report_value,
            "baseline_value": baseline_value,
            "delta": delta_value,
            "tolerance_used": tolerance_used,
            "blank_flag": blank_flag,
            "zero_flag": zero_flag,
            "abnormally_high_flag": high_flag,
            "delta_exceeds_tolerance_flag": delta_flag,
            "broken": is_broken,
        }
    )

result_df = pd.DataFrame(rows)
display(result_df)

broken_df = result_df[result_df["broken"] == True]
if broken_df.empty:
    print("\nValidation passed: no value anomalies detected.")
else:
    print(
        f"\nValidation failed: {len(broken_df)} visual(s) show potential anomalies."
    )
    display(broken_df)
    raise ValueError(
        "Potential broken report behavior detected by value anomaly checks."
    )